# Chapter 15 - Text Classification

Text classification is **supervised learning on text data**: given a document, predict
which category it belongs to. Spam detection, sentiment analysis, topic labelling, and
language identification are all text classification tasks.

The pipeline is straightforward: convert text to numerical features (as we did in the
previous notebook), then train a classifier. This notebook covers Naive Bayes — the
classic baseline for text — along with Logistic Regression and SVM, and shows how to
combine vectorization and classification into a single scikit-learn `Pipeline`.

**Topics covered:**
- Text classification as a supervised learning problem
- Bayes' theorem and the "naive" independence assumption
- MultinomialNB for text
- Complete pipeline: TfidfVectorizer -> MultinomialNB
- Scikit-learn Pipeline for text classification
- Sentiment analysis on synthetic data
- Evaluating text classifiers: confusion matrix, classification report
- Comparing classifiers: Naive Bayes vs Logistic Regression vs SVM
- GridSearchCV for joint vectorizer-classifier tuning
- Feature analysis: most informative features
- Summary: NLP workflow from raw text to predictions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score
)

np.random.seed(42)
%matplotlib inline

---
## 1. Text Classification as Supervised Learning

The general workflow:

```
Training documents + labels
        |
        v
  Vectorizer (text -> numbers)
        |
        v
  Feature matrix X, labels y
        |
        v
  Classifier.fit(X, y)
        |
        v
  New document -> Vectorizer.transform -> Classifier.predict -> Predicted label
```

The vectorizer must be fitted only on the training data. Using a `Pipeline` ensures
this happens correctly and prevents data leakage.

---
## 2. Synthetic Dataset: Sentiment Analysis

We will create a synthetic dataset of product reviews with positive and negative
sentiment labels. This avoids any external dependencies while giving us enough data
to demonstrate all the concepts.

In [ ]:
positive_reviews = [
    "This product is absolutely wonderful and exceeded my expectations.",
    "Excellent quality! I am very satisfied with this purchase.",
    "I love this item. It works perfectly and the price is great.",
    "Fantastic product. Highly recommend to anyone looking for quality.",
    "Amazing value for money. The build quality is superb.",
    "Very happy with my purchase. Fast shipping and great product.",
    "This is the best product I have bought this year. Outstanding.",
    "Great quality and excellent customer service. Will buy again.",
    "Absolutely love it. Perfect fit and beautiful design.",
    "Impressive performance and reliability. Highly satisfied.",
    "Wonderful product that delivers exactly what was promised.",
    "Superb craftsmanship. Worth every penny. Truly delighted.",
    "Everything about this product is perfect. Five stars deserved.",
    "Incredible quality at an affordable price. Very impressed.",
    "My favorite purchase this month. Works like a charm.",
    "Top notch quality and arrived quickly. Highly recommend.",
    "Brilliant product. Exactly what I needed. Very pleased.",
    "The quality surprised me in a good way. Excellent value.",
    "So happy I bought this. It is durable and well made.",
    "A perfect gift idea. The recipient was thrilled with it.",
    "Outstanding performance. Beats all competitors hands down.",
    "Premium quality material. Looks even better in person.",
    "Very reliable product. Have been using it daily with no issues.",
    "Smooth operation and intuitive design. Love this product.",
    "Exceeded all expectations. Best purchase I have made recently.",
]

negative_reviews = [
    "Terrible quality. Broke after just two days of use.",
    "Very disappointed with this purchase. Not worth the money.",
    "The product arrived damaged and customer service was unhelpful.",
    "Waste of money. The product does not work as advertised.",
    "Poor quality materials. Started falling apart within a week.",
    "Would not recommend. The product is cheaply made and flimsy.",
    "Horrible experience. Returning this product immediately.",
    "The product stopped working after one month. Very frustrating.",
    "Not as described. Much smaller and lower quality than expected.",
    "Extremely disappointed. This is the worst product I have ever bought.",
    "Save your money and buy something else. This is garbage.",
    "Terrible product. Does not function properly at all.",
    "Poorly designed and uncomfortable to use. Returning it.",
    "Broke on the first use. Absolutely terrible build quality.",
    "Complete waste of time and money. Avoid this product.",
    "Cheap materials and sloppy construction. Very unhappy.",
    "Does not match the description at all. Misleading listing.",
    "Awful product. I regret buying this. Huge disappointment.",
    "The worst purchase I have made. Nothing works correctly.",
    "Defective right out of the box. Requesting a refund.",
    "Flimsy and poorly made. Fell apart during first use.",
    "Do not waste your money on this. Total junk.",
    "Arrived late and broken. Customer support was useless.",
    "Very poor quality for the price. Expected much better.",
    "Stopped functioning after a few days. Cheaply manufactured.",
]

texts = positive_reviews + negative_reviews
labels = [1] * len(positive_reviews) + [0] * len(negative_reviews)

print(f"Total reviews: {len(texts)}")
print(f"Positive: {sum(labels)}, Negative: {len(labels) - sum(labels)}")

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")
print(f"Training positive: {sum(y_train)}, negative: {len(y_train) - sum(y_train)}")
print(f"Test positive:     {sum(y_test)}, negative: {len(y_test) - sum(y_test)}")

---
## 3. Bayes' Theorem: The Foundation of Naive Bayes

Given a document $d$ with features (words) $x_1, x_2, \ldots, x_n$, we want to find
the class $c$ that maximises the posterior probability:

$$P(c \mid x_1, x_2, \ldots, x_n) = \frac{P(x_1, x_2, \ldots, x_n \mid c) \cdot P(c)}{P(x_1, x_2, \ldots, x_n)}$$

The denominator $P(x_1, x_2, \ldots, x_n)$ is the same for all classes, so we only
need to compare the numerators:

$$P(c \mid \mathbf{x}) \propto P(\mathbf{x} \mid c) \cdot P(c)$$

- $P(c)$ is the **prior** — how common is class $c$ overall?
- $P(\mathbf{x} \mid c)$ is the **likelihood** — how probable is seeing these features
  given class $c$?

---
## 4. The "Naive" Assumption

Computing $P(x_1, x_2, \ldots, x_n \mid c)$ exactly is intractable — we would need to
estimate the joint probability of thousands of words. The "naive" assumption simplifies
this by assuming **all features are conditionally independent given the class**:

$$P(x_1, x_2, \ldots, x_n \mid c) = \prod_{i=1}^{n} P(x_i \mid c)$$

This is almost certainly wrong — words are not independent ("New" and "York" tend to
appear together). But in practice, this simplification works **surprisingly well** for
text classification because:

1. We only need to *rank* classes, not get calibrated probabilities.
2. The high dimensionality of text means errors in individual feature estimates tend to
   cancel out.
3. The model is extremely fast to train (just counting), which allows it to handle huge
   vocabularies.

---
## 5. MultinomialNB: Naive Bayes for Word Counts

For text, we use **Multinomial Naive Bayes**, which models the likelihood of each word
appearing a certain number of times. It works with both raw counts (BoW) and TF-IDF
weights (as long as they are non-negative).

The key parameter is **alpha** (Laplace smoothing): it adds a small count to every word
to avoid zero probabilities for words not seen in a particular class during training.

In [ ]:
# Step 1: Vectorize
tfidf = TfidfVectorizer(stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)  # transform only — no fitting on test data

print(f"Training feature matrix: {X_train_tfidf.shape}")
print(f"Test feature matrix:     {X_test_tfidf.shape}")

In [ ]:
# Step 2: Train MultinomialNB
nb = MultinomialNB(alpha=1.0)
nb.fit(X_train_tfidf, y_train)

# Step 3: Predict
y_pred_nb = nb.predict(X_test_tfidf)

print(f"Naive Bayes accuracy: {accuracy_score(y_test, y_pred_nb):.4f}")

In [ ]:
print(classification_report(y_test, y_pred_nb, target_names=['negative', 'positive']))

---
## 6. Using Scikit-Learn Pipeline

Manually calling `fit_transform` / `transform` is error-prone. A `Pipeline` chains
the vectorizer and classifier into a single estimator:

- `pipeline.fit(X_train_text, y_train)` — vectorizes and trains in one call
- `pipeline.predict(X_test_text)` — vectorizes and predicts in one call
- Works seamlessly with `cross_val_score` and `GridSearchCV`

The pipeline ensures the vectorizer is **only fitted on training data** during
cross-validation — preventing data leakage.

In [ ]:
pipe_nb = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', MultinomialNB(alpha=1.0)),
])

# Fit and predict in one line each
pipe_nb.fit(X_train, y_train)
y_pred_pipe = pipe_nb.predict(X_test)

print(f"Pipeline accuracy: {accuracy_score(y_test, y_pred_pipe):.4f}")

In [ ]:
# Cross-validation with the pipeline
cv_scores = cross_val_score(pipe_nb, texts, labels, cv=5, scoring='accuracy')
print(f"5-fold CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"Per-fold scores:    {cv_scores.round(4)}")

---
## 7. Confusion Matrix

The confusion matrix shows where the classifier makes mistakes. For a binary sentiment
task, the four cells represent true negatives, false positives, false negatives, and
true positives.

In [ ]:
cm = confusion_matrix(y_test, y_pred_pipe)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['negative', 'positive'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Naive Bayes Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Let us look at any misclassified reviews
misclassified = [(text, true, pred) 
                 for text, true, pred in zip(X_test, y_test, y_pred_pipe) 
                 if true != pred]

if misclassified:
    print(f"Misclassified reviews ({len(misclassified)}):")
    for text, true, pred in misclassified:
        label_map = {0: 'negative', 1: 'positive'}
        print(f"  True: {label_map[true]}, Predicted: {label_map[pred]}")
        print(f"  \"{text}\"\n")
else:
    print("All test reviews classified correctly!")

---
## 8. Comparing Classifiers: Naive Bayes vs Logistic Regression vs SVM

Three classifiers are commonly used for text:

| Classifier | Why it works for text |
|------------|----------------------|
| **MultinomialNB** | Extremely fast; designed for count/frequency data; strong baseline |
| **LogisticRegression** | Handles high-dimensional sparse data well; produces probabilities |
| **LinearSVC** | Effective in high-dimensional spaces; maximum-margin boundary |

All three work well with sparse, high-dimensional text features. Let us compare them.

In [ ]:
classifiers = {
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english')),
        ('clf', MultinomialNB(alpha=1.0)),
    ]),
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english')),
        ('clf', LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english')),
        ('clf', LinearSVC(max_iter=1000, random_state=42)),
    ]),
}

results = {}

for name, pipe in classifiers.items():
    cv = cross_val_score(pipe, texts, labels, cv=5, scoring='accuracy')
    results[name] = cv
    print(f"{name:25s}  CV accuracy: {cv.mean():.4f} (+/- {cv.std():.4f})")

In [ ]:
# Visualize the comparison
fig, ax = plt.subplots(figsize=(8, 5))

positions = range(len(results))
means = [v.mean() for v in results.values()]
stds = [v.std() for v in results.values()]

ax.bar(positions, means, yerr=stds, capsize=5, color=['steelblue', 'darkorange', 'seagreen'],
       edgecolor='k', alpha=0.8)
ax.set_xticks(positions)
ax.set_xticklabels(results.keys())
ax.set_ylabel('CV Accuracy')
ax.set_title('Classifier Comparison on Sentiment Data')
ax.set_ylim(0.5, 1.05)

for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + 0.02, f'{m:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 9. Detailed Evaluation of Each Classifier on the Test Set

Let us train each classifier on the training set and evaluate on the held-out test set
with a full classification report and confusion matrix.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, pipe) in zip(axes, classifiers.items()):
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['negative', 'positive'])
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    acc = accuracy_score(y_test, y_pred)
    ax.set_title(f'{name}\nAccuracy: {acc:.4f}')

plt.suptitle('Confusion Matrices — Test Set', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Full classification reports
for name, pipe in classifiers.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=['negative', 'positive']))
    print()

---
## 10. GridSearchCV: Tuning Vectorizer and Classifier Together

The beauty of the `Pipeline` is that we can tune parameters from **both** the vectorizer
and the classifier simultaneously using `GridSearchCV`. Parameter names use the `step__param`
convention:

- `tfidf__ngram_range` — n-gram range for the vectorizer
- `tfidf__max_df` — maximum document frequency
- `clf__alpha` — smoothing parameter for Naive Bayes

In [ ]:
pipe_tune = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', MultinomialNB()),
])

param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'tfidf__max_df': [0.8, 0.9, 1.0],
    'tfidf__max_features': [None, 50, 100],
    'clf__alpha': [0.01, 0.1, 0.5, 1.0, 2.0],
}

grid = GridSearchCV(pipe_tune, param_grid, cv=5, scoring='accuracy',
                    n_jobs=-1, return_train_score=True)
grid.fit(X_train, y_train)

print(f"Best parameters:  {grid.best_params_}")
print(f"Best CV accuracy: {grid.best_score_:.4f}")
print(f"Test accuracy:    {grid.score(X_test, y_test):.4f}")

In [ ]:
# Explore how alpha affects performance (holding other params at best values)
results_df = pd.DataFrame(grid.cv_results_)

# Filter to best non-alpha params
best_ngram = grid.best_params_['tfidf__ngram_range']
best_maxdf = grid.best_params_['tfidf__max_df']
best_maxfeat = grid.best_params_['tfidf__max_features']

mask = (
    (results_df['param_tfidf__ngram_range'] == best_ngram) &
    (results_df['param_tfidf__max_df'] == best_maxdf) &
    (results_df['param_tfidf__max_features'] == best_maxfeat)
)

alpha_results = results_df[mask].sort_values('param_clf__alpha')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(alpha_results['param_clf__alpha'].astype(float),
        alpha_results['mean_test_score'], 'o-', color='steelblue', linewidth=2, markersize=8)
ax.fill_between(
    alpha_results['param_clf__alpha'].astype(float),
    alpha_results['mean_test_score'] - alpha_results['std_test_score'],
    alpha_results['mean_test_score'] + alpha_results['std_test_score'],
    alpha=0.2, color='steelblue'
)
ax.set_xlabel('Alpha (smoothing parameter)')
ax.set_ylabel('CV Accuracy')
ax.set_title('Effect of Laplace Smoothing (alpha) on Naive Bayes')
ax.set_xscale('log')
plt.tight_layout()
plt.show()

---
## 11. GridSearchCV for Logistic Regression

We can do the same for Logistic Regression, tuning both vectorizer and regularization
strength C.

In [ ]:
pipe_lr = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000, random_state=42)),
])

param_grid_lr = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__max_df': [0.8, 1.0],
    'clf__C': [0.01, 0.1, 1.0, 10.0],
}

grid_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=5, scoring='accuracy', n_jobs=-1)
grid_lr.fit(X_train, y_train)

print(f"Best parameters:  {grid_lr.best_params_}")
print(f"Best CV accuracy: {grid_lr.best_score_:.4f}")
print(f"Test accuracy:    {grid_lr.score(X_test, y_test):.4f}")

---
## 12. Feature Analysis: Most Informative Features

One of the big advantages of linear models (Naive Bayes, Logistic Regression, Linear SVM)
is **interpretability**. We can look at the learned coefficients (weights) to see which
words are most strongly associated with each class.

For Logistic Regression and LinearSVC, positive coefficients push toward the positive
class, and negative coefficients push toward the negative class.

In [ ]:
# Train a Logistic Regression pipeline for feature analysis
pipe_analysis = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
])
pipe_analysis.fit(X_train, y_train)

feature_names = pipe_analysis.named_steps['tfidf'].get_feature_names_out()
coefficients = pipe_analysis.named_steps['clf'].coef_[0]

print(f"Number of features: {len(feature_names)}")
print(f"Coefficient range: [{coefficients.min():.4f}, {coefficients.max():.4f}]")

In [ ]:
# Top features for each class
n_top = 15
top_positive_idx = coefficients.argsort()[-n_top:][::-1]
top_negative_idx = coefficients.argsort()[:n_top]

print(f"Top {n_top} features for POSITIVE sentiment:")
for idx in top_positive_idx:
    print(f"  {feature_names[idx]:25s} coef: {coefficients[idx]:+.4f}")

print(f"\nTop {n_top} features for NEGATIVE sentiment:")
for idx in top_negative_idx:
    print(f"  {feature_names[idx]:25s} coef: {coefficients[idx]:+.4f}")

In [ ]:
# Visualize the most informative features
fig, ax = plt.subplots(figsize=(10, 8))

top_n = 12
top_pos_idx = coefficients.argsort()[-top_n:][::-1]
top_neg_idx = coefficients.argsort()[:top_n]

combined_idx = np.concatenate([top_neg_idx, top_pos_idx[::-1]])
combined_names = feature_names[combined_idx]
combined_coefs = coefficients[combined_idx]

colors = ['tomato' if c < 0 else 'seagreen' for c in combined_coefs]

ax.barh(range(len(combined_coefs)), combined_coefs, color=colors, edgecolor='k')
ax.set_yticks(range(len(combined_coefs)))
ax.set_yticklabels(combined_names)
ax.set_xlabel('Coefficient (weight)')
ax.set_title('Most Informative Features for Sentiment Classification\n'
             '(Logistic Regression coefficients)')
ax.axvline(x=0, color='black', linewidth=0.8)

# Add legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='tomato', edgecolor='k', label='Negative sentiment'),
    Patch(facecolor='seagreen', edgecolor='k', label='Positive sentiment'),
], loc='lower right')

plt.tight_layout()
plt.show()

The model has learned sensible patterns: words like "excellent", "love", "perfect", and
"great" push toward positive, while "waste", "terrible", "poor", and "broke" push toward
negative. This is a useful sanity check — if the top features do not make intuitive
sense, something may be wrong with the data or preprocessing.

---
## 13. Feature Analysis: Naive Bayes Log Probabilities

For Naive Bayes, we can examine the learned log-probabilities per class. The ratio
of log-probabilities between classes tells us how informative each feature is.

In [ ]:
pipe_nb_analysis = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', MultinomialNB(alpha=1.0)),
])
pipe_nb_analysis.fit(X_train, y_train)

nb_model = pipe_nb_analysis.named_steps['clf']
nb_features = pipe_nb_analysis.named_steps['tfidf'].get_feature_names_out()

# Log probability ratio: P(word|positive) / P(word|negative)
log_prob_ratio = nb_model.feature_log_prob_[1] - nb_model.feature_log_prob_[0]

top_pos = log_prob_ratio.argsort()[-10:][::-1]
top_neg = log_prob_ratio.argsort()[:10]

print("Words most indicative of POSITIVE (Naive Bayes):")
for idx in top_pos:
    print(f"  {nb_features[idx]:20s} log ratio: {log_prob_ratio[idx]:+.4f}")

print("\nWords most indicative of NEGATIVE (Naive Bayes):")
for idx in top_neg:
    print(f"  {nb_features[idx]:20s} log ratio: {log_prob_ratio[idx]:+.4f}")

---
## 14. Multi-Class Text Classification

Text classification extends naturally to multiple classes. Let us create a topic
classification dataset with three categories.

In [ ]:
sports_docs = [
    "The team won the championship after a thrilling final match.",
    "The goalkeeper made an incredible save in the last minute.",
    "The coach announced the new training schedule for the season.",
    "Players celebrated the victory with fans at the stadium.",
    "The league standings changed after this weekend results.",
    "The striker scored a hat trick in the cup semifinal.",
    "Injury concerns for the defender ahead of the tournament.",
    "The referee awarded a controversial penalty in the second half.",
    "The Olympic athlete broke the world record in the sprint.",
    "Fans traveled from across the country to support their team.",
]

tech_docs = [
    "The company released a new software update with enhanced features.",
    "Artificial intelligence is being used to improve healthcare outcomes.",
    "The startup raised funding to develop its cloud platform.",
    "Cybersecurity threats are increasing as more businesses go digital.",
    "The new processor delivers significant improvements in speed.",
    "Open source frameworks are accelerating software development.",
    "The database migration was completed without any downtime.",
    "Machine learning models are being deployed in production systems.",
    "The tech conference featured talks on blockchain and AI.",
    "Developers are adopting containerization for application deployment.",
]

food_docs = [
    "The restaurant serves fresh pasta made daily from scratch.",
    "Add the chopped vegetables to the pan and cook until tender.",
    "The bakery is famous for its sourdough bread and croissants.",
    "Season the fish with herbs and grill for ten minutes.",
    "The menu features a variety of dishes from local farms.",
    "Whisk the eggs and sugar together until light and fluffy.",
    "The chef recommends pairing the steak with a red wine sauce.",
    "Fresh herbs add wonderful flavor to any salad dressing.",
    "The recipe requires two cups of flour and baking powder.",
    "Slow cooking brings out the deep rich flavors of the stew.",
]

mc_texts = sports_docs + tech_docs + food_docs
mc_labels = ['sports'] * 10 + ['technology'] * 10 + ['food'] * 10

print(f"Total documents: {len(mc_texts)}")
print(f"Classes: {sorted(set(mc_labels))}")

In [ ]:
mc_X_train, mc_X_test, mc_y_train, mc_y_test = train_test_split(
    mc_texts, mc_labels, test_size=0.3, random_state=42, stratify=mc_labels
)

mc_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2))),
    ('clf', MultinomialNB(alpha=0.5)),
])

mc_pipe.fit(mc_X_train, mc_y_train)
mc_pred = mc_pipe.predict(mc_X_test)

print(f"Multi-class accuracy: {accuracy_score(mc_y_test, mc_pred):.4f}")
print()
print(classification_report(mc_y_test, mc_pred))

In [ ]:
# Multi-class confusion matrix
cm_mc = confusion_matrix(mc_y_test, mc_pred, labels=['food', 'sports', 'technology'])

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_mc,
                               display_labels=['food', 'sports', 'technology'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Multi-class Topic Classification\n(Naive Bayes)')
plt.tight_layout()
plt.show()

---
## 15. Predicting New Documents

Once trained, the pipeline can classify entirely new text.

In [ ]:
new_docs = [
    "The forward scored a last minute goal to win the game.",
    "Mix the butter and sugar together and fold in the flour.",
    "The new API allows developers to integrate machine learning models.",
    "Season the chicken with paprika and roast for forty minutes.",
]

predictions = mc_pipe.predict(new_docs)

for doc, pred in zip(new_docs, predictions):
    print(f"  [{pred:10s}] {doc}")

---
## 16. Comparing All Three Classifiers on Multi-Class Data

In [ ]:
mc_classifiers = {
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2))),
        ('clf', MultinomialNB(alpha=0.5)),
    ]),
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2))),
        ('clf', LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2))),
        ('clf', LinearSVC(max_iter=1000, random_state=42)),
    ]),
}

mc_results = {}
for name, pipe in mc_classifiers.items():
    cv = cross_val_score(pipe, mc_texts, mc_labels, cv=5, scoring='accuracy')
    mc_results[name] = cv
    print(f"{name:25s}  CV accuracy: {cv.mean():.4f} (+/- {cv.std():.4f})")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

positions = range(len(mc_results))
means = [v.mean() for v in mc_results.values()]
stds = [v.std() for v in mc_results.values()]

ax.bar(positions, means, yerr=stds, capsize=5,
       color=['steelblue', 'darkorange', 'seagreen'], edgecolor='k', alpha=0.8)
ax.set_xticks(positions)
ax.set_xticklabels(mc_results.keys())
ax.set_ylabel('CV Accuracy')
ax.set_title('Multi-class Topic Classification — Classifier Comparison')
ax.set_ylim(0.5, 1.05)

for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + 0.02, f'{m:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 17. The Effect of N-gram Range on Classification

Let us see how adding bigrams and trigrams affects classifier performance.

In [ ]:
ngram_configs = [(1, 1), (1, 2), (1, 3), (2, 2)]
ngram_scores = {}

for ngram in ngram_configs:
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=ngram)),
        ('clf', MultinomialNB(alpha=0.5)),
    ])
    cv = cross_val_score(pipe, texts, labels, cv=5, scoring='accuracy')
    ngram_scores[str(ngram)] = cv
    
    # Also check feature count
    pipe.fit(texts, labels)
    n_features = len(pipe.named_steps['tfidf'].vocabulary_)
    print(f"ngram_range={str(ngram):8s}  features: {n_features:4d}  "
          f"CV accuracy: {cv.mean():.4f} (+/- {cv.std():.4f})")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

labels_plot = list(ngram_scores.keys())
means_plot = [v.mean() for v in ngram_scores.values()]
stds_plot = [v.std() for v in ngram_scores.values()]

ax.bar(range(len(labels_plot)), means_plot, yerr=stds_plot, capsize=5,
       color='steelblue', edgecolor='k', alpha=0.8)
ax.set_xticks(range(len(labels_plot)))
ax.set_xticklabels(labels_plot)
ax.set_xlabel('ngram_range')
ax.set_ylabel('CV Accuracy')
ax.set_title('Effect of N-gram Range on Sentiment Classification')
ax.set_ylim(0.5, 1.05)

plt.tight_layout()
plt.show()

---
## 18. Making Predictions with Probabilities

Both MultinomialNB and LogisticRegression support `predict_proba`, which returns the
estimated probability of each class. This is useful when you need confidence levels.

In [ ]:
pipe_proba = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', MultinomialNB(alpha=1.0)),
])
pipe_proba.fit(X_train, y_train)

test_sentences = [
    "This product is excellent and works perfectly.",
    "Terrible quality. Waste of money.",
    "The product is okay. Nothing special.",
    "Absolutely love it. Best purchase ever.",
    "Broke after one day. Very disappointed.",
]

probabilities = pipe_proba.predict_proba(test_sentences)
predictions = pipe_proba.predict(test_sentences)

label_map = {0: 'negative', 1: 'positive'}
print(f"{'Sentence':55s} {'Prediction':12s} {'P(neg)':8s} {'P(pos)':8s}")
print('-' * 85)
for sent, pred, prob in zip(test_sentences, predictions, probabilities):
    print(f"{sent:55s} {label_map[pred]:12s} {prob[0]:.4f}   {prob[1]:.4f}")

---
## 19. The Complete NLP Workflow Diagram

Here is the full pipeline from raw text to predictions:

```
1. Collect labelled text data
       |
2. Train/test split (on raw text)
       |
3. Build a Pipeline:
       |--- TfidfVectorizer (preprocessing + feature extraction)
       |        - stop_words, ngram_range, max_features, max_df
       |--- Classifier (MultinomialNB / LogisticRegression / LinearSVC)
       |        - alpha / C / other hyperparameters
       |
4. Tune with GridSearchCV (cross-validation on training set)
       |
5. Evaluate on test set (classification report, confusion matrix)
       |
6. Inspect feature weights (sanity check)
       |
7. Deploy: pipeline.predict(new_text)
```

---
## 20. End-to-End Example: Spam Detection

Let us put the entire workflow together one more time on a different task: classifying
messages as spam or not spam (ham).

In [ ]:
ham_messages = [
    "Hey, are we still meeting for lunch tomorrow?",
    "Can you send me the project report by Friday?",
    "Happy birthday! Hope you have a wonderful day.",
    "The meeting has been moved to 3pm in the conference room.",
    "Thanks for your help with the presentation yesterday.",
    "Do you want to grab coffee after work today?",
    "I will be running late for dinner tonight. Sorry!",
    "Can you pick up some groceries on your way home?",
    "The kids have a school event next Wednesday morning.",
    "Just checking in to see how your interview went.",
    "Please review the attached document when you get a chance.",
    "Looking forward to seeing you at the weekend barbecue.",
    "The flight is confirmed for next Thursday at 9am.",
    "Remember to bring your laptop to the workshop tomorrow.",
    "Great job on the quarterly report. Well done.",
]

spam_messages = [
    "CONGRATULATIONS! You have won a free iPhone! Click here now!",
    "Buy cheap medications online! Discount prices guaranteed!",
    "You are selected for a cash prize of ten thousand dollars!",
    "FREE entry to win a luxury holiday! Text WIN to 12345!",
    "Earn money from home! No experience needed! Act now!",
    "Limited time offer! Buy one get one free on all products!",
    "Claim your free gift card worth five hundred dollars today!",
    "Make thousands of dollars per week working from home!",
    "You have been chosen to receive a special discount offer!",
    "Act now to receive your exclusive bonus prize immediately!",
    "Get rich quick with this amazing investment opportunity!",
    "Free trial! No credit card required! Sign up instantly!",
    "Winner notification! You have won the grand prize lottery!",
    "Double your income guaranteed! Click the link below!",
    "Special promotion just for you! Unbeatable prices inside!",
]

spam_texts = ham_messages + spam_messages
spam_labels = ['ham'] * len(ham_messages) + ['spam'] * len(spam_messages)

print(f"Total messages: {len(spam_texts)} (ham: {len(ham_messages)}, spam: {len(spam_messages)})")

In [ ]:
# Split
sp_X_train, sp_X_test, sp_y_train, sp_y_test = train_test_split(
    spam_texts, spam_labels, test_size=0.3, random_state=42, stratify=spam_labels
)

# Pipeline with grid search
spam_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', MultinomialNB()),
])

spam_grid = GridSearchCV(
    spam_pipe,
    param_grid={
        'tfidf__ngram_range': [(1, 1), (1, 2)],
        'clf__alpha': [0.01, 0.1, 0.5, 1.0],
    },
    cv=3, scoring='accuracy', n_jobs=-1
)
spam_grid.fit(sp_X_train, sp_y_train)

print(f"Best params:    {spam_grid.best_params_}")
print(f"Best CV score:  {spam_grid.best_score_:.4f}")
print(f"Test accuracy:  {spam_grid.score(sp_X_test, sp_y_test):.4f}")

In [ ]:
# Confusion matrix
sp_pred = spam_grid.predict(sp_X_test)

fig, ax = plt.subplots(figsize=(6, 5))
cm_sp = confusion_matrix(sp_y_test, sp_pred, labels=['ham', 'spam'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm_sp, display_labels=['ham', 'spam'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Spam Detection — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(sp_y_test, sp_pred, target_names=['ham', 'spam']))

In [ ]:
# Classify new messages
new_messages = [
    "Hey, do you want to go for a run this evening?",
    "YOU HAVE WON! Claim your free prize now! Text WINNER!",
    "The deadline for the report is next Monday.",
    "Get rich instantly! No investment needed! Click here!",
]

spam_predictions = spam_grid.predict(new_messages)

for msg, pred in zip(new_messages, spam_predictions):
    print(f"  [{pred:4s}] {msg}")

---
## 21. Summary

| Concept | Key takeaway |
|---------|-------------|
| **Text classification** | Supervised learning on vectorized text; the pipeline is Vectorizer -> Classifier |
| **Bayes' theorem** | $P(class \mid features) \propto P(features \mid class) \times P(class)$ |
| **Naive assumption** | Features (words) are conditionally independent given the class — wrong but effective |
| **MultinomialNB** | Fast, simple, strong baseline for text; use alpha for smoothing |
| **Pipeline** | Chains vectorizer + classifier; ensures correct fit/transform during cross-validation |
| **Logistic Regression** | Handles sparse, high-dimensional text well; interpretable coefficients |
| **LinearSVC** | Maximum-margin classifier; often competitive with logistic regression on text |
| **GridSearchCV** | Tune vectorizer and classifier parameters jointly using `step__param` names |
| **Feature analysis** | Inspect top coefficients or log-probability ratios to understand what the model learned |

The NLP workflow:
1. Collect and label data
2. Split into train/test
3. Build a Pipeline (TfidfVectorizer + classifier)
4. Tune hyperparameters with GridSearchCV
5. Evaluate on the test set
6. Inspect features for sanity
7. Deploy